In [1]:
import requests  # Para llamar a las APIs
import pandas as pd  # Para poder convertir los datos en df
import mysql.connector  # Para conectar Python con MySQL
from mysql.connector import Error  # Para importar los errores de SQL
import os   # Para crear variables de entorno y poder mantener la contraseña secreta
from dotenv import load_dotenv  # Para leer el archivo .env de la variable de entorno
load_dotenv()  # Carga el contenido del .env
password_sql = os.getenv("pass_sql") # Para acceder a la contraseña que hemos indicado en el .env
import numpy as np  # Para poder hacer el clean

En lo referente a la obtención de datos, el repositorio incluye únicamente el código de extracción de algunas artistas representativas, ya que se considera suficiente para mostrar el funcionamiento del proceso completo de extracción y tratamiento de datos.

Para obtener información del resto de artistas, bastaría con modificar el nombre de la cantante en las funciones ya desarrolladas.

Somos conscientes de que el proceso podría haberse optimizado mediante una función automatizada capaz de extraer los datos de todas las artistas de forma simultánea, reduciendo considerablemente la cantidad de código repetido. Sin embargo, se optó por una aproximación más manual y controlada debido a limitaciones de tiempo y para minimizar el riesgo de pérdida de datos ante posibles fallos de ejecución, cierres inesperados de Python o problemas del equipo durante la extracción masiva de información.

Por ello, se priorizó la estabilidad del proceso y la correcta obtención de los datos frente a una automatización más compleja.

In [ ]:
# Sacamos los datos de la API de Deezer

import requests
import pandas as pd

def conseguir_canciones(artist_name, limit=50):     # Función para conseguir 50 canciones de la artista que quieras
    url = "https://api.deezer.com/search"           # URL base de la API para buscar
    params = {                                      # Para insertar los parámetros de la búsqueda
        "q": f'artist:"{artist_name}"',             # La API busca artista por q
        "limit": limit                              # Limit sirve para que saque solo 50
    }
    try:                                            # Bloque try/except para llamar a la API y que nos dé el resultado
        response = requests.get(url, params=params) # Llamada a la API con los parámetros que hemos indicado
        if response.status_code != 200:             # Si el código de respuesta no es 200, entonces se ha producido un error
            print("Error en la llamada a Deezer:", response.status_code)    # Si falla lo va a informar
            return pd.DataFrame([])                                         # Y también nos va a devolver el DF vacío
        data = response.json()                                              # Si no falla da la respuesta en json
    except Error as e:                                                      # Si se produce un error en la llamada a la API, lo captura y lo muestra
        print("Error en la llamada a Deezer:", e)                           # Para que nos diga exactamente cuál es el error
        return pd.DataFrame([])                                            

    canciones = data.get("data", [])                # Obtiene la lista de canciones
    lista_canciones = []                            # Lista donde se van a acumular las filas para el DF final

    for cancion in canciones:                       # Bucle para que itere con todas las canciones y nos vaya dando los datos
        track_id = cancion["id"]                    # Nos da el id de la canción
        track = requests.get(f"https://api.deezer.com/track/{track_id}").json()  # Pide todos los datos que estén en track
        album = requests.get(f"https://api.deezer.com/album/{track['album']['id']}").json() # Pide todos los datos que estén en album

        genre_id = album.get("genre_id")   # Extrae el ID del género de album
        genre_name = None                  # Creamos la variable "vacía"
        if album.get("genres") and album["genres"].get("data"):   # Le decimoa que en el caso de que genres exista en album
            genre_name = album["genres"]["data"][0].get("name")   # Entonces nos diga el primer género que aparezca (por si hay varios)

        tracks_type = "colaboracion" if len(track.get("contributors", [])) > 1 else album.get("record_type", "track")
        # Buscamos colaboraciones. Para ello, si en contributor aparece más de 1, entonces se trata de una colaboración
        # Si no, le decimos que nos ponga lo que aparezca en record_type que es single o album, y si no que ponga track
        release_year = track.get("release_date", "")[:4] if track.get("release_date") else None
        # Obtiene el año de lanzamiento solo si hay 4 cifras, si no pondrá None 
        lista_canciones.append({                   # Creamos un diccionario para añadir todos los datos que necesitamos
            "id_artista": track["artist"]["id"],
            "nombre_artista": artist_name,
            "titulo_cancion": track["title"],
            "titulo_album": track["album"]["title"],
            "tipo": tracks_type,
            "año_lanzamiento": release_year,
            "genero": genre_name,
            "id_genero": genre_id,
        })

    return pd.DataFrame(lista_canciones)        # Hacemos que nos devuelta el resultado en un DF

df_artista = conseguir_canciones("Björk", limit=50)  # Llamamos a la función con el nombre de la artista que queremos y el número de canciones que queremos obtener
df_artista                                           # Y lo mostramos para comprobar que se han obtenido los datos correctamente
df_artista.to_csv("Björk.csv", index=False, encoding="utf-8") # Guardamos el DF en un CSV para poder hacer el clean posteriormente


In [ ]:
df_artista = conseguir_canciones("Destiny's Child", limit=50)
df_artista
df_artista.to_csv("Destiny's Child.csv", index=False, encoding="utf-8")

In [ ]:
df_artista = conseguir_canciones("Missy Elliott", limit=50)
df_artista
df_artista.to_csv("Missy Elliott.csv", index=False, encoding="utf-8")

In [ ]:
df_artista = conseguir_canciones("Alanis Morissette", limit=50)
df_artista
df_artista.to_csv("Alanis Morissette.csv", index=False, encoding="utf-8")

In [ ]:
df_artista = conseguir_canciones("Spice Girls", limit=50)
df_artista
df_artista.to_csv("Spice Girls.csv", index=False, encoding="utf-8")

In [ ]:
df_artista = conseguir_canciones("Shakira", limit=50)
df_artista
df_artista.to_csv("Shakira.csv", index=False, encoding="utf-8")

In [ ]:
df_artista = conseguir_canciones("Jennifer Lopez", limit=50)
df_artista
df_artista.to_csv("Jennifer Lopez.csv", index=False, encoding="utf-8")

In [ ]:
df_artista = conseguir_canciones("Courtney Love", limit=50)
df_artista
df_artista.to_csv("Courtney Love.csv", index=False, encoding="utf-8")

In [ ]:
df_artista = conseguir_canciones("Amy Winehouse", limit=50)
df_artista
df_artista.to_csv("Amy Winehouse.csv", index=False, encoding="utf-8")

In [ ]:
df_artista = conseguir_canciones("TLC", limit=50)
df_artista
df_artista.to_csv("TLC.csv", index=False, encoding="utf-8")

In [ ]:
df_artista = conseguir_canciones("Janet Jackson", limit=50)
df_artista
df_artista.to_csv("Janet Jackson.csv", index=False, encoding="utf-8")

In [ ]:
df_artista = conseguir_canciones("Cesária Évora", limit=50)
df_artista
df_artista.to_csv("Cesária Évora.csv", index=False, encoding="utf-8")

In [22]:
df_artista = conseguir_canciones("Sinéad O'Connor", limit=50)
df_artista
df_artista.to_csv("Sinéad O'Connor.csv", index=False, encoding="utf-8")

In [ ]:
# Convertimos los datos de cada artista a csv

artista1 = pd.read_csv("Björk.csv")
artista2 = pd.read_csv("Destiny's Child.csv")
artista3 = pd.read_csv("Missy Elliott.csv")
artista4 = pd.read_csv("Alanis Morissette.csv")
artista5 = pd.read_csv("Spice Girls.csv")
artista6 = pd.read_csv("Shakira.csv")
artista7 = pd.read_csv("Jennifer Lopez.csv")
artista8 = pd.read_csv("Courtney Love.csv")
artista9 = pd.read_csv("Amy Winehouse.csv")
artista10 = pd.read_csv("TLC.csv")
artista11 = pd.read_csv("Janet Jackson.csv")
artista12 = pd.read_csv("Cesária Évora.csv")
artista13 = pd.read_csv("Sinéad O'Connor.csv")
artistas_completo_deezer = pd.concat([artista1, artista2, artista3, artista4, artista5, artista6, artista7, artista8, artista9, artista10, artista11, artista12, artista13], ignore_index=True)
artistas_completo_deezer.to_csv("artistas_completo.csv", index=False, encoding="utf-8")

In [ ]:
# Sacamos los datos de la API de Last FM

BASE_URL = "http://ws.audioscrobbler.com/2.0/"   # URL base de la API de Last.fm.
API_KEY = "1c3f93bc38412b6ffa077ff634369d02"     # API Key personal para poder acceder a la API.

def buscar_artista(nombre_artista):     # Definimos una función para buscar artista.
    params = {                          # Parámetros que se le van a pasar a la API para que nos dé la información del artista que queremos
        "method": "artist.getInfo",     # El método que se le va a pasar a la API para que nos dé la información del artista que queremos
        "artist": nombre_artista,       # El nombre del artista que queremos buscar
        "api_key": API_KEY,             # La API Key para poder acceder a la API
        "format": "json"                # El formato en el que queremos que nos devuelva la información (en este caso, json)
    }
    try:                                # Bloque try/except para llamar a la API y que nos dé el resultado
        response = requests.get(BASE_URL, params=params)    # Llamada a la API con los parámetros que hemos definido
        datos = response.json()                             # Si no falla da la respuesta en json
        artista = datos["artist"]                           # Extraemos la información del artista que nos interesa
        nombre = artista["name"]                            # Extraemos el nombre del artista
        biografia = artista["bio"]["summary"]               # Extraemos la biografía del artista
        listeners = artista["stats"]["listeners"]           # Extraemos el número de oyentes del artista
        playcount = artista["stats"]["playcount"]           # Extraemos el número de reproducciones del artista
        similares = artista["similar"]["artist"]            # Extraemos la información de los artistas similares al artista que hemos buscado
        artistas_similares = []                             # Creamos una lista vacía para acumular los nombres de los artistas similares
        for artista_similar in similares:                   # Bucle para que itere con todos los artistas similares y nos vaya dando los nombres
            artistas_similares.append(artista_similar["name"])  # Añadimos el nombre de cada artista similar a la lista que hemos creado
        return {                                            # Devolvemos un diccionario con toda la información que hemos extraído del artista que hemos buscado
            "nombre_artista": nombre,                       # El nombre del artista
            "biografia": biografia,                         # La biografía del artista
            "listeners": listeners,                         # El número de oyentes del artista
            "playcount": playcount,                         # El número de reproducciones del artista
            "artistas_similares": ", ".join(artistas_similares)  # Los nombres de los artistas similares separados por comas
        }
    except:                                                 # Si falla lo va a informar
        print(f"Error al buscar {nombre_artista}")          # Nos informa del error indicando el nombre del artista que hemos intentado buscar
        return None                                         # Devolvemos None para indicar que no se ha podido obtener la información del artista

datos_artista = buscar_artista("Björk")                     # Llamamos a la función para buscar la información de Björk y guardamos el resultado en una variable
df_LFM = pd.DataFrame([datos_artista])                      # Convertimos el resultado en un DataFrame para poder trabajar con él de forma más cómoda
df_LFM                                                      # Mostramos el DataFrame para comprobar que se ha obtenido la información correctamente
df_LFM.to_csv("Björk_LFM.csv", index=False, encoding="utf-8")  # Guardamos el DataFrame en un archivo CSV para poder acceder a él más adelante

In [ ]:
datos_artista = buscar_artista("Destiny's Child")
df_LFM = pd.DataFrame([datos_artista])
df_LFM
df_LFM.to_csv("Destiny's Child_LFM.csv", index=False, encoding="utf-8")

In [ ]:
datos_artista = buscar_artista("Missy Elliott")
df_LFM = pd.DataFrame([datos_artista])
df_LFM
df_LFM.to_csv("Missy Elliott_LFM.csv", index=False, encoding="utf-8")

In [ ]:
datos_artista = buscar_artista("Alanis Morissette")
df_LFM = pd.DataFrame([datos_artista])
df_LFM
df_LFM.to_csv("Alanis Morissette_LFM.csv", index=False, encoding="utf-8")

In [ ]:
datos_artista = buscar_artista("Spice Girls")
df_LFM = pd.DataFrame([datos_artista])
df_LFM
df_LFM.to_csv("Spice Girls_LFM.csv", index=False, encoding="utf-8")

In [ ]:
datos_artista = buscar_artista("Shakira")
df_LFM = pd.DataFrame([datos_artista])
df_LFM
df_LFM.to_csv("Shakira_LFM.csv", index=False, encoding="utf-8")

In [ ]:
datos_artista = buscar_artista("Jennifer Lopez")
df_LFM = pd.DataFrame([datos_artista])
df_LFM
df_LFM.to_csv("Jennifer Lopez_LFM.csv", index=False, encoding="utf-8")

In [ ]:
datos_artista = buscar_artista("Courtney Love")
df_LFM = pd.DataFrame([datos_artista])
df_LFM
df_LFM.to_csv("Courtney Love_LFM.csv", index=False, encoding="utf-8")

In [ ]:
datos_artista = buscar_artista("Amy Winehouse")
df_LFM = pd.DataFrame([datos_artista])
df_LFM
df_LFM.to_csv("Amy Winehouse_LFM.csv", index=False, encoding="utf-8")

In [ ]:
datos_artista = buscar_artista("TLC")
df_LFM = pd.DataFrame([datos_artista])
df_LFM
df_LFM.to_csv("TLC_LFM.csv", index=False, encoding="utf-8")

In [ ]:
datos_artista = buscar_artista("Janet Jackson")
df_LFM = pd.DataFrame([datos_artista])
df_LFM
df_LFM.to_csv("Janet Jackson_LFM.csv", index=False, encoding="utf-8")

In [ ]:
datos_artista = buscar_artista("Cesária Évora")
df_LFM = pd.DataFrame([datos_artista])
df_LFM
df_LFM.to_csv("Cesária Évora_LFM.csv", index=False, encoding="utf-8")

In [ ]:
datos_artista = buscar_artista("Sinéad O'Connor")
df_LFM = pd.DataFrame([datos_artista])
df_LFM
df_LFM.to_csv("Sinéad O'Connor_LFM.csv", index=False, encoding="utf-8")

In [ ]:
# Convertimos los datos de cada artista a csv

artista1 = pd.read_csv("Björk_LFM.csv")
artista2 = pd.read_csv("Destiny's Child_LFM.csv")
artista3 = pd.read_csv("Missy Elliott_LFM.csv")
artista4 = pd.read_csv("Alanis Morissette_LFM.csv")
artista5 = pd.read_csv("Spice Girls_LFM.csv")
artista6 = pd.read_csv("Shakira_LFM.csv")
artista7 = pd.read_csv("Jennifer Lopez_LFM.csv")
artista8 = pd.read_csv("Courtney Love_LFM.csv")
artista9 = pd.read_csv("Amy Winehouse_LFM.csv")
artista10 = pd.read_csv("TLC_LFM.csv")
artista11 = pd.read_csv("Janet Jackson_LFM.csv")
artista12 = pd.read_csv("Cesária Évora_LFM.csv")
artista13 = pd.read_csv("Sinéad O'Connor_LFM.csv")
artistas_completo_LFM = pd.concat([artista1, artista2, artista3, artista4, artista5, artista6, artista7, artista8, artista9, artista10, artista11, artista12, artista13], ignore_index=True)
artistas_completo_LFM.to_csv("artistas_completo_LFM.csv", index=False, encoding="utf-8")

In [2]:
# Conexión a MySQL

try:
    connection = mysql.connector.connect (
    host = "127.0.0.1", 
    user = "root",
    password = password_sql, 
    )
    print ("Conexion exitosa")
except Error as e:
    print ("Ha ocurrido un error: {e}")


Conexion exitosa


In [3]:
#Creación Base de Datos

try:
    cursor = connection.cursor()
    query_crear_bbdd = """
    CREATE DATABASE IF NOT EXISTS musicstream
    CHARACTER SET utf8mb4
    COLLATE utf8mb4_unicode_ci
    """
    cursor.execute(query_crear_bbdd)
    print("Query existosa")
except Error as e:
    print(e)

Query existosa


In [3]:
cursor = connection.cursor()
cursor.execute("use musicstream")

In [5]:
df_artistas = pd.read_csv("artistas_completo_deezer.csv",
                 encoding="utf-8")

print(df_artistas.info())

<class 'pandas.DataFrame'>
RangeIndex: 2012 entries, 0 to 2011
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Unnamed: 0       1100 non-null   float64
 1   id_artista       2012 non-null   int64  
 2   nombre_artista   2012 non-null   str    
 3   titulo_cancion   2012 non-null   str    
 4   titulo_album     2012 non-null   str    
 5   tipo             2012 non-null   str    
 6   año_lanzamiento  2012 non-null   int64  
 7   genero           1880 non-null   str    
 8   id_genero        2007 non-null   float64
dtypes: float64(2), int64(2), str(5)
memory usage: 265.1 KB
None


In [4]:
cursor.execute("use musicstream")
query_crear_tabla = '''CREATE TABLE artistas (
                        id_artista INT PRIMARY KEY AUTO_INCREMENT, 
                        nombre_artista VARCHAR(50) NOT NULL
                        );'''
cursor.execute(query_crear_tabla)
print ("Tabla creada correctamente")

Tabla creada correctamente


In [5]:
df_tabla_artistas = pd.read_csv(
    "artistas_completo_deezer.csv",
    encoding="utf-8"
)
df_tabla_artistas = df_tabla_artistas[["nombre_artista"]].drop_duplicates()
for index, fila in df_tabla_artistas.iterrows():
    query_insert = """
    INSERT INTO artistas (nombre_artista)
    VALUES (%s)
    """
    valores = (fila["nombre_artista"],)
    cursor.execute(query_insert, valores)
connection.commit()
print("Datos insertados correctamente")

Datos insertados correctamente


In [6]:
cursor.execute("use musicstream")
query_crear_tabla = """
CREATE TABLE canciones (
    id_cancion INT PRIMARY KEY AUTO_INCREMENT,
    id_artista INT NOT NULL,
    titulo_cancion VARCHAR(255) NOT NULL,
    titulo_album VARCHAR(255),
    tipo VARCHAR(100),
    año_lanzamiento INT,
    genero VARCHAR(100),
    id_genero FLOAT,
    FOREIGN KEY (id_artista)
    REFERENCES artistas(id_artista)
);
"""
cursor.execute(query_crear_tabla)
print("Tabla canciones creada correctamente")

Tabla canciones creada correctamente


In [9]:
df_LFM = pd.read_csv("artistas_completo_deezer.csv",
                 encoding="utf-8")

print(df_LFM.info())

<class 'pandas.DataFrame'>
RangeIndex: 2012 entries, 0 to 2011
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Unnamed: 0       1100 non-null   float64
 1   id_artista       2012 non-null   int64  
 2   nombre_artista   2012 non-null   str    
 3   titulo_cancion   2012 non-null   str    
 4   titulo_album     2012 non-null   str    
 5   tipo             2012 non-null   str    
 6   año_lanzamiento  2012 non-null   int64  
 7   genero           1880 non-null   str    
 8   id_genero        2007 non-null   float64
dtypes: float64(2), int64(2), str(5)
memory usage: 265.1 KB
None


In [7]:
df_tabla_canciones = pd.read_csv(
    "artistas_completo_deezer.csv",
    encoding="utf-8"
)
for index, fila in df_tabla_canciones.iterrows():
    query_id_artista = """
    SELECT id_artista
    FROM artistas
    WHERE nombre_artista = %s
    """
    cursor.execute(
        query_id_artista,
        (fila["nombre_artista"],)
    )
    resultado = cursor.fetchone()
    id_artista = resultado[0]
    query_insert = """
    INSERT INTO canciones (
        id_artista,
        titulo_cancion,
        titulo_album,
        tipo,
        año_lanzamiento,
        genero,
        id_genero
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    """
    valores = (
        id_artista,
        fila["titulo_cancion"],
        fila["titulo_album"],
        fila["tipo"],
        fila["año_lanzamiento"],
        fila["genero"]
        if pd.notna(fila["genero"])
        else None,
        fila["id_genero"]
        if pd.notna(fila["id_genero"])
        else None
    )
    cursor.execute(query_insert, valores)
connection.commit()
print("Datos insertados correctamente")

Datos insertados correctamente


In [8]:
query_crear_tabla = """
CREATE TABLE informacion_artistas (
    id_info INT PRIMARY KEY AUTO_INCREMENT,
    id_artista INT NOT NULL,
    nombre_artista VARCHAR(50),
    biografia TEXT,
    listeners BIGINT,
    playcount BIGINT,
    artistas_similares TEXT,
    FOREIGN KEY (id_artista)
    REFERENCES artistas(id_artista)
);
"""
cursor.execute(query_crear_tabla)
print("Tabla informacion_artistas creada correctamente")

Tabla informacion_artistas creada correctamente


In [12]:
df_lfm = pd.read_csv(
    "artistas_completo_LFM.csv",
    encoding="utf-8"
)
for index, fila in df_lfm.iterrows():
    query_id_artista = """
    SELECT id_artista
    FROM artistas
    WHERE nombre_artista = %s
    """
    cursor.execute(
        query_id_artista,
        (fila["nombre_artista"],)
    )
    resultado = cursor.fetchone()
    if resultado is None:
        print(f"Artista {fila["nombre_artista"]} no encontrado en tabla 'artistas'. Omitiendo.")
        continue
    id_artista = resultado[0]
    query_insert = """
    INSERT INTO informacion_artistas (
        id_artista,
        nombre_artista,
        biografia,
        listeners,
        playcount,
        artistas_similares
    )
    VALUES (%s, %s, %s, %s, %s, %s)
    """
    valores = (
        id_artista,
        fila["nombre_artista"],
        fila["biografia"] if pd.notna(fila["biografia"]) else None,
        fila["listeners"] if pd.notna(fila["listeners"]) else None,
        fila["playcount"] if pd.notna(fila["playcount"]) else None,
        fila["artistas_similares"] if pd.notna(fila["artistas_similares"]) else None
    )
    cursor.execute(query_insert, valores)
connection.commit()
print("Datos insertados correctamente")


Artista Rocío Dúrcal no encontrado en tabla 'artistas'. Omitiendo.
Datos insertados correctamente


In [19]:
# ¿Qué género musical ha dominado cada década?

query_genero_dominante = """
SELECT FLOOR(año_lanzamiento / 10) * 10 AS decada, genero,
COUNT(*) AS total_canciones
FROM canciones
WHERE genero IS NOT NULL
GROUP BY decada, genero
ORDER BY decada ASC, total_canciones DESC
"""
cursor.execute(query_genero_dominante)
resultado = cursor.fetchall()
decadas_vistas = []
lista_resultados = []
for fila in resultado:
    decada = fila[0]
    if decada not in decadas_vistas:
        lista_resultados.append({
            "decada": f"{fila[0]}s",
            "genero_dominante": fila[1],
            "cantidad_canciones": fila[2]
        })
        decadas_vistas.append(decada)
df_generos_dominantes = pd.DataFrame(lista_resultados)
df_generos_dominantes

,decada,genero_dominante,cantidad_canciones
0,1960s,Rock,11
1,1970s,Rock,24
2,1980s,Pop,59
3,1990s,Pop,118
4,2000s,Pop,228
5,2010s,Pop,271
6,2020s,Pop,313


In [ ]:
# Quién tiene la audiencia más fiel? Listeners vs playcounts

query_reproducciones = """
SELECT nombre_artista, listeners, playcount,
ROUND(playcount / listeners, 0) AS reproducciones_por_pax
FROM informacion_artistas
ORDER BY reproducciones_por_pax DESC;
"""
cursor.execute(query_reproducciones)
resultado = cursor.fetchall()
df_reproducciones = pd.DataFrame(
    resultado,
    columns=[
        "nombre_artista",
        "listeners",
        "playcount",
        "reproducciones_por_pax"
    ]
)
df_reproducciones

,nombre_artista,listeners,playcount,reproducciones_por_pax
0,Taylor Swift,5984147,3712355647,620
1,Billie Eilish,4274273,848588063,199
2,Beyonce,6428520,707170351,110
3,Dua Lipa,3319054,323557741,97
4,Björk,3486294,262188907,75
5,Madonna,5707724,371493131,65
6,Aitana,250392,15253397,61
7,Adele,5660677,321186884,57
8,Karol G,1468751,77700895,53
9,Amy Winehouse,4664433,206178952,44


In [ ]:
# En qué década las artistas femeninas lanzaron más álbumes?

query_albumes_decada = """
SELECT FLOOR(año_lanzamiento / 10) * 10 AS decada,
COUNT(DISTINCT titulo_album) AS total_albumes
FROM canciones
GROUP BY decada
ORDER BY total_albumes DESC;
"""
cursor.execute(query_albumes_decada)
resultado = cursor.fetchall()
df_albumes_decada = pd.DataFrame(
    resultado,
    columns=[
        "decada",
        "total_albumes"
    ]
)


,decada,total_albumes
0,2020,381
1,2010,325
2,2000,182
3,1990,76
4,1980,37
5,1960,9
6,1970,9


In [ ]:
# Cómo ha cambiado el playcount entre artistas de diferentes épocas?

query_evolucion_artistas = """
SELECT i.nombre_artista,
MIN(c.año_lanzamiento) AS primer_lanzamiento, i.playcount
FROM informacion_artistas i
INNER JOIN canciones c
ON i.id_artista = c.id_artista
GROUP BY i.nombre_artista, i.playcount
ORDER BY MIN(c.año_lanzamiento) ASC;
"""
cursor.execute(query_evolucion_artistas)
resultado = cursor.fetchall()
df_evolucion_artistas = pd.DataFrame(
    resultado,
    columns=[
        "nombre_artista",
        "primer_lanzamiento",
        "playcount"
    ]
)


,nombre_artista,primer_lanzamiento,playcount
0,Rita Pavone,1961,555307
1,Aretha Franklin,1962,53166557
2,Sylvie Vartan,1962,1951209
3,Janis Joplin,1967,34110921
4,Nina Simone,1969,75272531
5,Kate Bush,1978,147523239
6,Gloria Gaynor,1982,11672851
7,Bonnie Tyler,1983,21852788
8,Cyndi Lauper,1983,38868049
9,Diana Ross,1984,21055145


In [ ]:
# En qué décadas se colaboraba más y menos?

query_colaboraciones_decada = """
SELECT FLOOR(año_lanzamiento / 10) * 10 AS decada,
COUNT(*) AS num_colaboraciones
FROM canciones
WHERE tipo = 'colaboración'
GROUP BY decada
ORDER BY decada;
"""
cursor.execute(query_colaboraciones_decada)
resultado = cursor.fetchall()
df_colaboraciones_decada = pd.DataFrame(
    resultado,
    columns=[
        "decada",
        "num_colaboraciones"
    ]
)


,decada,num_colaboraciones
0,1960,2
1,1970,3
2,1980,7
3,1990,22
4,2000,87
5,2010,181
6,2020,266


In [ ]:
# Cuántos géneros distintos abarca cada artista?

query_generos_artista = """
SELECT a.nombre_artista,
COUNT(DISTINCT c.genero) AS num_generos
FROM artistas a
JOIN canciones c
ON a.id_artista = c.id_artista
GROUP BY a.id_artista, a.nombre_artista
ORDER BY num_generos DESC;
"""
cursor.execute(query_generos_artista)
resultado = cursor.fetchall()
df_generos_artista = pd.DataFrame(
    resultado,
    columns=[
        "nombre_artista",
        "num_generos"
    ]
)


,nombre_artista,num_generos
0,Shakira,9
1,TLC,9
2,Karina,9
3,Massiel,8
4,Nathy Peluso,7
5,Courtney Love,6
6,Missy Elliott,6
7,Jennifer Lopez,6
8,Diana Ross,6
9,Karol G,5


In [ ]:
# Qué artistas han colaborado más con otros artistas a lo largo de su carrera?

query_colaboraciones = """
SELECT a.nombre_artista,
COUNT(*) AS total_colaboraciones
FROM artistas a
JOIN canciones c
ON a.id_artista = c.id_artista
WHERE c.tipo = 'colaboracion'
GROUP BY a.nombre_artista
ORDER BY total_colaboraciones DESC;
"""
cursor.execute(query_colaboraciones)
resultado = cursor.fetchall()
df_colaboraciones = pd.DataFrame(
    resultado,
    columns=[
        "nombre_artista",
        "total_colaboraciones"
    ]
)



,nombre_artista,total_colaboraciones
0,Missy Elliott,41
1,Jennifer Lopez,35
2,Lola Indigo,34
3,Shakira,34
4,Karol G,33
5,Karina,31
6,Aitana,29
7,Celia Cruz,26
8,Mercedes Sosa,24
9,Cesária Évora,20
